# Q-Learning from Scratch: Robot in a Maze

This notebook implements Q-learning completely from scratch — no RL libraries.

**What you will build:**
1. A configurable grid maze environment
2. A Q-learning agent from scratch
3. Training loop with epsilon decay
4. Visualizations: reward curves, Q-table heatmaps, learned policy arrows
5. Evaluation: watch the trained agent solve the maze

## Cell 1 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('All imports successful.')

## Cell 2 — Define the Maze Environment

The maze is a 2D grid:
- `0` = open cell
- `1` = wall (blocked)
- `2` = goal

**State encoding:** Each (row, col) cell is flattened to a single integer index `state = row * ncols + col`.

**Reward structure:**
- Hitting a wall or boundary: `-5` (strong discouragement)
- Each valid step: `-1` (encourages finding the *shortest* path)
- Reaching the goal: `+100`

In [ ]:
class MazeEnv:
    """
    A simple grid-based maze environment for Q-learning.
    
    Grid encoding:
        0 = open cell
        1 = wall
        2 = goal
    
    Actions: 0=Up, 1=Down, 2=Left, 3=Right
    """
    
    ACTION_UP    = 0
    ACTION_DOWN  = 1
    ACTION_LEFT  = 2
    ACTION_RIGHT = 3
    ACTION_NAMES = ['Up', 'Down', 'Left', 'Right']
    ACTION_DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # (row_delta, col_delta)
    
    # Rewards
    REWARD_GOAL    =  100
    REWARD_STEP    =   -1
    REWARD_WALL    =   -5

    def __init__(self, grid, start):
        """
        Parameters
        ----------
        grid  : 2D list/array, 0=open, 1=wall, 2=goal
        start : (row, col) tuple for the robot's start position
        """
        self.grid  = np.array(grid)
        self.nrows, self.ncols = self.grid.shape
        self.start = start
        
        # Locate goal
        goal_positions = list(zip(*np.where(self.grid == 2)))
        assert len(goal_positions) == 1, "Maze must have exactly one goal cell."
        self.goal = tuple(goal_positions[0])
        
        self.n_states  = self.nrows * self.ncols
        self.n_actions = 4
        
        self.current_pos = start
    
    # ---------------------------------------------------------------
    # Core interface
    # ---------------------------------------------------------------
    
    def reset(self):
        """Reset robot to start. Returns initial state index."""
        self.current_pos = self.start
        return self._pos_to_state(self.current_pos)
    
    def step(self, action):
        """
        Take one action.
        Returns (next_state, reward, done)
        """
        dr, dc = self.ACTION_DELTAS[action]
        new_row = self.current_pos[0] + dr
        new_col = self.current_pos[1] + dc
        
        # Check boundary and wall collision
        if self._is_blocked(new_row, new_col):
            # Robot stays in place but receives wall penalty
            reward = self.REWARD_WALL
            done   = False
            return self._pos_to_state(self.current_pos), reward, done
        
        # Valid move
        self.current_pos = (new_row, new_col)
        
        if self.current_pos == self.goal:
            reward = self.REWARD_GOAL
            done   = True
        else:
            reward = self.REWARD_STEP
            done   = False
        
        return self._pos_to_state(self.current_pos), reward, done
    
    # ---------------------------------------------------------------
    # Helpers
    # ---------------------------------------------------------------
    
    def _pos_to_state(self, pos):
        """Convert (row, col) → flat state index."""
        return pos[0] * self.ncols + pos[1]
    
    def _state_to_pos(self, state):
        """Convert flat state index → (row, col)."""
        return (state // self.ncols, state % self.ncols)
    
    def _is_blocked(self, row, col):
        """True if out of bounds or a wall."""
        if row < 0 or row >= self.nrows or col < 0 or col >= self.ncols:
            return True
        return self.grid[row, col] == 1
    
    def render(self):
        """Print a simple ASCII representation of the maze."""
        symbols = {0: '.', 1: '#', 2: 'G'}
        for r in range(self.nrows):
            row_str = ''
            for c in range(self.ncols):
                if (r, c) == self.current_pos:
                    row_str += 'R '
                elif (r, c) == self.start:
                    row_str += 'S '
                else:
                    row_str += symbols[self.grid[r, c]] + ' '
            print(row_str)
        print()


# ---------------------------------------------------------------
# Define our maze
# ---------------------------------------------------------------
#
#   S . . #
#   . # . .
#   . . . .
#   # . . G
#
MAZE_GRID = [
    [0, 0, 0, 1],
    [0, 1, 0, 0],
    [0, 0, 0, 0],
    [1, 0, 0, 2],
]
START = (0, 0)

env = MazeEnv(MAZE_GRID, START)
print(f'Maze shape : {env.nrows} x {env.ncols}')
print(f'State space: {env.n_states} states')
print(f'Action space: {env.n_actions} actions')
print(f'Start: {env.start}  |  Goal: {env.goal}')
print()
print('Initial maze (R=Robot, S=Start, G=Goal, #=Wall):')
env.render()

## Cell 3 — Visualize the Maze

In [ ]:
def plot_maze(env, title='Maze'):
    fig, ax = plt.subplots(figsize=(5, 5))
    
    cmap = plt.cm.colors.ListedColormap(['white', 'black', 'limegreen'])
    ax.imshow(env.grid, cmap=cmap, vmin=0, vmax=2)
    
    # Draw grid lines
    for x in range(env.ncols + 1):
        ax.axvline(x - 0.5, color='gray', linewidth=0.8)
    for y in range(env.nrows + 1):
        ax.axhline(y - 0.5, color='gray', linewidth=0.8)
    
    # Annotate cells
    for r in range(env.nrows):
        for c in range(env.ncols):
            if (r, c) == env.start:
                ax.text(c, r, 'S', ha='center', va='center', fontsize=16,
                        fontweight='bold', color='blue')
            elif (r, c) == env.goal:
                ax.text(c, r, 'G', ha='center', va='center', fontsize=16,
                        fontweight='bold', color='white')
            elif env.grid[r, c] == 1:
                ax.text(c, r, '■', ha='center', va='center', fontsize=14, color='white')
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(range(env.ncols))
    ax.set_yticks(range(env.nrows))
    ax.set_xticklabels([f'c{i}' for i in range(env.ncols)])
    ax.set_yticklabels([f'r{i}' for i in range(env.nrows)])
    plt.tight_layout()
    plt.show()

plot_maze(env, 'Our 4×4 Maze (S=Start, G=Goal, ■=Wall)')

## Cell 4 — The Q-Table: Initialize and Inspect

The Q-table is a 2D NumPy array of shape `(n_states, n_actions)`.

All values start at 0 — the agent knows nothing.

In [ ]:
# Initialize Q-table: shape (16 states, 4 actions)
Q = np.zeros((env.n_states, env.n_actions))

print('Q-table shape:', Q.shape)
print('Q-table (initial — all zeros):')
print()

# Pretty print as a pandas-style table
import sys

header = f"{'State':>8}  {'Up':>8}  {'Down':>8}  {'Left':>8}  {'Right':>8}"
print(header)
print('-' * len(header))
for s in range(env.n_states):
    pos = env._state_to_pos(s)
    cell_type = {0: 'open', 1: 'wall', 2: 'goal'}[env.grid[pos]]
    row = f"s{s:02d}{pos!s:>6}  {Q[s,0]:>8.2f}  {Q[s,1]:>8.2f}  {Q[s,2]:>8.2f}  {Q[s,3]:>8.2f}   ({cell_type})"
    print(row)

## Cell 5 — The Q-Learning Agent

This class encapsulates:
- **Epsilon-greedy action selection** (exploration vs exploitation)
- **The Bellman update equation** (the core of Q-learning)
- **Epsilon decay** (shift from exploration to exploitation over time)

```
Q(s, a) ← Q(s, a) + α × [r + γ × max_a' Q(s', a') − Q(s, a)]
```

In [ ]:
class QLearningAgent:
    """
    A tabular Q-learning agent.
    
    Implements:
    - Epsilon-greedy action selection
    - Bellman Q-update
    - Epsilon decay schedule
    """
    
    def __init__(
        self,
        n_states,
        n_actions,
        alpha=0.1,          # learning rate
        gamma=0.99,         # discount factor
        epsilon=1.0,        # initial exploration rate
        epsilon_min=0.01,   # minimum exploration rate
        epsilon_decay=0.995 # multiplicative decay per episode
    ):
        self.n_states    = n_states
        self.n_actions   = n_actions
        self.alpha       = alpha
        self.gamma       = gamma
        self.epsilon     = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        
        # The Q-table: initialized to zeros
        self.Q = np.zeros((n_states, n_actions))
    
    # ---------------------------------------------------------------
    # Action selection: Epsilon-Greedy
    # ---------------------------------------------------------------
    
    def select_action(self, state):
        """
        With probability epsilon: explore (random action)
        Otherwise:                exploit (greedy — best known action)
        """
        if np.random.rand() < self.epsilon:
            # EXPLORE: random action
            return np.random.randint(self.n_actions)
        else:
            # EXPLOIT: action with highest Q-value
            return np.argmax(self.Q[state])
    
    # ---------------------------------------------------------------
    # Core update: Bellman equation
    # ---------------------------------------------------------------
    
    def update(self, state, action, reward, next_state, done):
        """
        Apply the Q-learning update rule:
        
        Q(s, a) ← Q(s, a) + α × [r + γ × max_a' Q(s', a') − Q(s, a)]
        
        If done (terminal state), future reward is 0.
        """
        current_q  = self.Q[state, action]
        
        # Best Q-value from the next state (0 if terminal)
        max_next_q = 0 if done else np.max(self.Q[next_state])
        
        # Target = immediate reward + discounted future value
        target     = reward + self.gamma * max_next_q
        
        # TD error = how wrong was our current estimate?
        td_error   = target - current_q
        
        # Update: move current estimate toward target by a fraction alpha
        self.Q[state, action] = current_q + self.alpha * td_error
    
    # ---------------------------------------------------------------
    # Epsilon decay: call at end of each episode
    # ---------------------------------------------------------------
    
    def decay_epsilon(self):
        """Reduce epsilon toward epsilon_min."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    # ---------------------------------------------------------------
    # Policy extraction
    # ---------------------------------------------------------------
    
    def get_policy(self):
        """Return the greedy policy: best action for each state."""
        return np.argmax(self.Q, axis=1)


# Instantiate agent
agent = QLearningAgent(
    n_states     = env.n_states,
    n_actions    = env.n_actions,
    alpha        = 0.2,
    gamma        = 0.99,
    epsilon      = 1.0,
    epsilon_min  = 0.01,
    epsilon_decay= 0.995
)

print('Agent initialized.')
print(f'  Learning rate (α) : {agent.alpha}')
print(f'  Discount factor (γ): {agent.gamma}')
print(f'  Initial epsilon (ε): {agent.epsilon}')
print(f'  Epsilon decay      : {agent.epsilon_decay}')
print(f'  Q-table shape      : {agent.Q.shape}')

## Cell 6 — Step-by-Step Trace: One Update (Demystifying the Math)

Before training, let's manually execute one Q-update to see the math concretely.

In [ ]:
# Manual one-step trace
print('=== MANUAL Q-UPDATE TRACE ===')
print()

# Setup
s  = env.reset()           # Start state
a  = env.ACTION_RIGHT      # Manually choose: move Right from (0,0)
s_prime, r, done = env.step(a)

pos_s       = env._state_to_pos(s)
pos_s_prime = env._state_to_pos(s_prime)

current_q  = agent.Q[s, a]
max_next_q = np.max(agent.Q[s_prime])
target     = r + agent.gamma * max_next_q
td_error   = target - current_q
new_q      = current_q + agent.alpha * td_error

print(f'State s        : {s}  → position {pos_s}')
print(f'Action a       : {a}  → {env.ACTION_NAMES[a]}')
print(f'Next state s\' : {s_prime} → position {pos_s_prime}')
print(f'Reward r       : {r}')
print(f'Done?          : {done}')
print()
print('--- Q-Update Math ---')
print(f'Current Q(s,a)          = {current_q:.4f}')
print(f'max Q(s\', all actions)  = {max_next_q:.4f}')
print(f'Target = r + γ·max_Q    = {r} + {agent.gamma} × {max_next_q:.4f} = {target:.4f}')
print(f'TD error = target - Q   = {target:.4f} - {current_q:.4f} = {td_error:.4f}')
print(f'New Q = Q + α·TD_error  = {current_q:.4f} + {agent.alpha} × {td_error:.4f} = {new_q:.4f}')
print()
print('(Since all Q-values are 0 initially, new Q = α × r)')
print(f'= {agent.alpha} × {r} = {agent.alpha * r}')

## Cell 7 — Training Loop

This is the full Q-learning training loop. We run for 3000 episodes.

Each episode:
1. Reset robot to start
2. Loop until goal or max steps:
   - ε-greedy action selection
   - Environment step
   - Q-update
3. Decay epsilon

In [ ]:
# Re-initialize for a clean training run
np.random.seed(42)
env   = MazeEnv(MAZE_GRID, START)
agent = QLearningAgent(
    n_states     = env.n_states,
    n_actions    = env.n_actions,
    alpha        = 0.2,
    gamma        = 0.99,
    epsilon      = 1.0,
    epsilon_min  = 0.01,
    epsilon_decay= 0.995
)

# Training hyperparameters
N_EPISODES = 3000
MAX_STEPS  = 200   # Max steps per episode (prevents infinite loops)

# Tracking metrics
episode_rewards  = []
episode_steps    = []
epsilon_history  = []
success_history  = []   # 1 if goal reached, 0 otherwise

print(f'Training for {N_EPISODES} episodes...')
print()

for episode in range(N_EPISODES):
    state       = env.reset()
    total_reward = 0
    steps        = 0
    success      = False
    
    for step in range(MAX_STEPS):
        # 1. Select action using ε-greedy
        action = agent.select_action(state)
        
        # 2. Take action in environment
        next_state, reward, done = env.step(action)
        
        # 3. Update Q-table (THE CORE STEP)
        agent.update(state, action, reward, next_state, done)
        
        # 4. Advance state
        state        = next_state
        total_reward += reward
        steps        += 1
        
        if done:
            success = True
            break
    
    # 5. Decay epsilon at end of episode
    agent.decay_epsilon()
    
    # Track metrics
    episode_rewards.append(total_reward)
    episode_steps.append(steps)
    epsilon_history.append(agent.epsilon)
    success_history.append(int(success))
    
    # Print progress every 500 episodes
    if (episode + 1) % 500 == 0:
        recent_success = np.mean(success_history[-100:]) * 100
        recent_reward  = np.mean(episode_rewards[-100:])
        recent_steps   = np.mean(episode_steps[-100:])
        print(f'Episode {episode+1:4d}/{N_EPISODES} | '
              f'ε={agent.epsilon:.3f} | '
              f'Success (last 100): {recent_success:.0f}% | '
              f'Avg reward: {recent_reward:7.1f} | '
              f'Avg steps: {recent_steps:.1f}')

print()
print('Training complete!')

## Cell 8 — Training Curves

Four plots:
1. **Episode Reward** — should trend upward as agent learns
2. **Episode Steps** — should trend downward (shorter paths)
3. **Success Rate** (rolling 100 ep) — should approach 100%
4. **Epsilon Decay** — shows the exploration → exploitation shift

In [ ]:
def smooth(data, window=50):
    """Rolling mean for smoothing noisy curves."""
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Q-Learning Training Curves', fontsize=15, fontweight='bold')

episodes = np.arange(N_EPISODES)

# 1. Episode reward
ax = axes[0, 0]
ax.plot(episodes, episode_rewards, alpha=0.3, color='steelblue', label='Raw')
ax.plot(np.arange(len(smooth(episode_rewards))), smooth(episode_rewards),
        color='steelblue', linewidth=2, label='Smoothed (50-ep)')
ax.set_title('Episode Reward')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Episode steps
ax = axes[0, 1]
ax.plot(episodes, episode_steps, alpha=0.3, color='coral', label='Raw')
ax.plot(np.arange(len(smooth(episode_steps))), smooth(episode_steps),
        color='coral', linewidth=2, label='Smoothed (50-ep)')
ax.set_title('Steps per Episode (lower = better)')
ax.set_xlabel('Episode')
ax.set_ylabel('Steps')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Success rate (rolling)
ax = axes[1, 0]
rolling_success = [np.mean(success_history[max(0, i-100):i+1]) * 100
                   for i in range(len(success_history))]
ax.plot(episodes, rolling_success, color='green', linewidth=2)
ax.axhline(100, color='green', linestyle='--', alpha=0.4, label='100%')
ax.set_title('Success Rate (rolling 100 episodes)')
ax.set_xlabel('Episode')
ax.set_ylabel('Success Rate (%)')
ax.set_ylim(0, 105)
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Epsilon decay
ax = axes[1, 1]
ax.plot(episodes, epsilon_history, color='purple', linewidth=2)
ax.fill_between(episodes, epsilon_history, alpha=0.15, color='purple')
ax.axhline(agent.epsilon_min, color='red', linestyle='--', label=f'ε_min={agent.epsilon_min}')
ax.set_title('Epsilon Decay (Exploration → Exploitation)')
ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon (ε)')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 9 — Q-Table Heatmaps

Visualize the learned Q-values for each of the 4 actions across the maze grid.

**What to look for:**
- Cells near the goal should have **high Q-values**
- Walls are shown as gray (they have Q-values but the agent learns not to go there)
- The optimal path should have consistently higher values than detours

In [ ]:
def plot_q_heatmaps(env, Q, action_names):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Learned Q-Values by Action', fontsize=14, fontweight='bold')
    
    # Reshape Q for grid display
    Q_grid = Q.reshape(env.nrows, env.ncols, env.n_actions)
    
    for a, ax in enumerate(axes.flat):
        q_map = Q_grid[:, :, a].copy()
        
        # Mask walls
        wall_mask = (env.grid == 1)
        q_map_masked = np.ma.masked_where(wall_mask, q_map)
        
        im = ax.imshow(q_map_masked, cmap='RdYlGn', interpolation='nearest')
        plt.colorbar(im, ax=ax)
        
        # Annotate with Q-values
        for r in range(env.nrows):
            for c in range(env.ncols):
                if env.grid[r, c] == 1:
                    ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1,
                                 color='black', zorder=2))
                    ax.text(c, r, '■', ha='center', va='center',
                            color='white', fontsize=12, zorder=3)
                elif (r, c) == env.start:
                    ax.text(c, r, f'S\n{Q[env._pos_to_state((r,c)), a]:.1f}',
                            ha='center', va='center', fontsize=9, color='blue', fontweight='bold')
                elif (r, c) == env.goal:
                    ax.text(c, r, f'G\n{Q[env._pos_to_state((r,c)), a]:.1f}',
                            ha='center', va='center', fontsize=9, color='darkgreen', fontweight='bold')
                else:
                    ax.text(c, r, f'{Q[env._pos_to_state((r,c)), a]:.1f}',
                            ha='center', va='center', fontsize=9)
        
        ax.set_title(f'Q(state, {action_names[a]})', fontsize=12)
        ax.set_xticks(range(env.ncols))
        ax.set_yticks(range(env.nrows))
        ax.set_xticklabels([f'c{i}' for i in range(env.ncols)])
        ax.set_yticklabels([f'r{i}' for i in range(env.nrows)])
    
    plt.tight_layout()
    plt.show()

plot_q_heatmaps(env, agent.Q, env.ACTION_NAMES)

## Cell 10 — Learned Policy: Arrow Map

Extract the greedy policy (best action per state) and visualize it as arrows on the maze.

Each arrow shows: *"From this cell, the agent has learned to move in this direction."*

In [ ]:
def plot_policy(env, Q, title='Learned Policy'):
    policy = np.argmax(Q, axis=1)  # Best action per state
    
    # Arrow directions: (dx, dy) for matplotlib quiver
    # matplotlib y-axis is inverted for imshow, so Up = negative dy
    arrow_dirs = {
        env.ACTION_UP:    (0, -0.4),
        env.ACTION_DOWN:  (0,  0.4),
        env.ACTION_LEFT:  (-0.4, 0),
        env.ACTION_RIGHT: (0.4,  0),
    }
    
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # Draw maze background
    display_grid = np.where(env.grid == 1, -1, 0).astype(float)
    display_grid[env.goal] = 1
    
    cmap = LinearSegmentedColormap.from_list('maze', ['black', 'lightyellow', 'limegreen'])
    ax.imshow(display_grid, cmap=cmap, vmin=-1, vmax=1)
    
    # Grid lines
    for x in range(env.ncols + 1):
        ax.axvline(x - 0.5, color='gray', linewidth=0.8)
    for y in range(env.nrows + 1):
        ax.axhline(y - 0.5, color='gray', linewidth=0.8)
    
    # Draw arrows
    for r in range(env.nrows):
        for c in range(env.ncols):
            if env.grid[r, c] == 1:
                ax.text(c, r, '■', ha='center', va='center',
                        fontsize=14, color='white', zorder=3)
                continue
            if (r, c) == env.goal:
                ax.text(c, r, '★\nGOAL', ha='center', va='center',
                        fontsize=11, color='white', fontweight='bold', zorder=3)
                continue
            if (r, c) == env.start:
                label = 'S'
                ax.text(c - 0.35, r - 0.35, label, ha='center', va='center',
                        fontsize=8, color='blue', fontweight='bold', zorder=4)
            
            s  = env._pos_to_state((r, c))
            a  = policy[s]
            dx, dy = arrow_dirs[a]
            ax.annotate('', xy=(c + dx, r + dy), xytext=(c, r),
                        arrowprops=dict(arrowstyle='->', color='darkblue',
                                       lw=2.0), zorder=3)
            # Q-value annotation
            max_q = np.max(Q[s])
            ax.text(c, r + 0.3, f'{max_q:.0f}', ha='center', va='center',
                    fontsize=7, color='darkred', zorder=4)
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(range(env.ncols))
    ax.set_yticks(range(env.nrows))
    ax.set_xticklabels([f'c{i}' for i in range(env.ncols)])
    ax.set_yticklabels([f'r{i}' for i in range(env.nrows)])
    
    legend_elements = [
        mpatches.Patch(color='lightyellow', label='Open cell (number = max Q-value)'),
        mpatches.Patch(color='black', label='Wall'),
        mpatches.Patch(color='limegreen', label='Goal'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()

plot_policy(env, agent.Q, 'Learned Policy: Optimal Direction from Each Cell')

## Cell 11 — Evaluate: Watch the Trained Agent

Run the trained agent greedily (no exploration) and print the path it takes.

In [ ]:
def run_episode_greedy(env, agent, max_steps=50, verbose=True):
    """Run one episode with pure greedy policy (no exploration)."""
    state = env.reset()
    path  = [env._state_to_pos(state)]
    actions_taken = []
    total_reward  = 0
    
    for step in range(max_steps):
        # Pure greedy: no randomness
        action = np.argmax(agent.Q[state])
        next_state, reward, done = env.step(action)
        
        pos = env._state_to_pos(next_state)
        path.append(pos)
        actions_taken.append(env.ACTION_NAMES[action])
        total_reward += reward
        state = next_state
        
        if verbose:
            print(f'  Step {step+1:2d}: action={env.ACTION_NAMES[action]:5s} '
                  f'→ pos={pos}  reward={reward:4d}  Q-values={agent.Q[state].round(1)}')
        
        if done:
            break
    
    return path, actions_taken, total_reward, done


print('=== TRAINED AGENT: GREEDY EVALUATION ===')
print(f'Start: {env.start}  →  Goal: {env.goal}')
print()
path, actions, total_reward, reached_goal = run_episode_greedy(env, agent)
print()
print(f'Path taken    : {" → ".join([str(p) for p in path])}')
print(f'Actions taken : {" → ".join(actions)}')
print(f'Steps         : {len(actions)}')
print(f'Total reward  : {total_reward}')
print(f'Goal reached  : {reached_goal}')

## Cell 12 — Visualize the Agent's Path

In [ ]:
def plot_path(env, path, title='Agent Path'):
    fig, ax = plt.subplots(figsize=(6, 6))
    
    display_grid = np.where(env.grid == 1, -1, 0).astype(float)
    display_grid[env.goal] = 1
    
    cmap = LinearSegmentedColormap.from_list('maze', ['black', 'lightyellow', 'limegreen'])
    ax.imshow(display_grid, cmap=cmap, vmin=-1, vmax=1)
    
    for x in range(env.ncols + 1):
        ax.axvline(x - 0.5, color='gray', linewidth=0.8)
    for y in range(env.nrows + 1):
        ax.axhline(y - 0.5, color='gray', linewidth=0.8)
    
    # Draw walls
    for r in range(env.nrows):
        for c in range(env.ncols):
            if env.grid[r, c] == 1:
                ax.text(c, r, '■', ha='center', va='center',
                        fontsize=14, color='white', zorder=3)
    
    # Draw path
    n = len(path)
    for i, (r, c) in enumerate(path):
        if i == 0:
            color = 'blue'
            label = 'S'
        elif i == n - 1:
            color = 'green'
            label = '★'
        else:
            color = 'royalblue'
            label = str(i)
        
        circle = plt.Circle((c, r), 0.25, color=color, zorder=4, alpha=0.8)
        ax.add_patch(circle)
        ax.text(c, r, label, ha='center', va='center',
                fontsize=9, color='white', fontweight='bold', zorder=5)
    
    # Draw path lines
    for i in range(len(path) - 1):
        r1, c1 = path[i]
        r2, c2 = path[i+1]
        ax.plot([c1, c2], [r1, r2], 'b--', linewidth=1.5, alpha=0.5, zorder=3)
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(range(env.ncols))
    ax.set_yticks(range(env.nrows))
    ax.set_xticklabels([f'c{i}' for i in range(env.ncols)])
    ax.set_yticklabels([f'r{i}' for i in range(env.nrows)])
    plt.tight_layout()
    plt.show()

plot_path(env, path, f'Trained Agent Path: {len(path)-1} steps to Goal')

## Cell 13 — Ablation: Effect of Learning Rate and Discount Factor

Let's run Q-learning with different hyperparameters and compare success rates.

This gives you intuition for **why hyperparameter choices matter**.

In [ ]:
def train_agent(alpha, gamma, n_episodes=1500, verbose=False):
    np.random.seed(42)
    env_local   = MazeEnv(MAZE_GRID, START)
    agent_local = QLearningAgent(
        n_states=env_local.n_states, n_actions=env_local.n_actions,
        alpha=alpha, gamma=gamma, epsilon=1.0,
        epsilon_min=0.01, epsilon_decay=0.995
    )
    rewards  = []
    successes = []
    for ep in range(n_episodes):
        state = env_local.reset()
        total_r = 0
        success = False
        for _ in range(200):
            a = agent_local.select_action(state)
            ns, r, done = env_local.step(a)
            agent_local.update(state, a, r, ns, done)
            state = ns; total_r += r
            if done: success = True; break
        agent_local.decay_epsilon()
        rewards.append(total_r)
        successes.append(int(success))
    return rewards, successes


configs = [
    {'alpha': 0.05, 'gamma': 0.99, 'label': 'α=0.05 (slow learner)'},
    {'alpha': 0.20, 'gamma': 0.99, 'label': 'α=0.20 (balanced)'},
    {'alpha': 0.80, 'gamma': 0.99, 'label': 'α=0.80 (aggressive)'},
    {'alpha': 0.20, 'gamma': 0.50, 'label': 'α=0.20 γ=0.50 (shortsighted)'},
    {'alpha': 0.20, 'gamma': 0.99, 'label': 'α=0.20 γ=0.99 (farsighted)'},
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hyperparameter Ablation Study', fontsize=13, fontweight='bold')

colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

for cfg, color in zip(configs, colors):
    rewards, successes = train_agent(cfg['alpha'], cfg['gamma'])
    
    # Smooth reward
    sm_r = smooth(rewards, window=50)
    ax1.plot(sm_r, label=cfg['label'], color=color, linewidth=2)
    
    # Rolling success rate
    rolling = [np.mean(successes[max(0,i-100):i+1])*100 for i in range(len(successes))]
    ax2.plot(rolling, label=cfg['label'], color=color, linewidth=2)
    
    print(f"{cfg['label']:40s} | Final success rate: {np.mean(successes[-200:])*100:.0f}%")

ax1.set_title('Smoothed Episode Reward')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.set_title('Rolling Success Rate (100-ep window)')
ax2.set_xlabel('Episode'); ax2.set_ylabel('Success Rate (%)')
ax2.set_ylim(0, 105)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 14 — Final Q-Table Inspection

Print the learned Q-table in a readable format to see the actual values that drive the policy.

In [ ]:
print('=== FINAL LEARNED Q-TABLE ===')
print()
print(f"{'State':>10}  {'Up':>8}  {'Down':>8}  {'Left':>8}  {'Right':>8}  {'BestAction':>12}  Type")
print('-' * 80)

policy = agent.get_policy()

for s in range(env.n_states):
    pos  = env._state_to_pos(s)
    cell = env.grid[pos]
    cell_type = {0: 'open', 1: 'WALL', 2: 'GOAL'}[cell]
    best_a = policy[s]
    best_a_name = env.ACTION_NAMES[best_a]
    
    # Highlight best Q-value in each row
    q_vals = agent.Q[s]
    row = f"s{s:02d} {str(pos):>6}  "
    for a in range(env.n_actions):
        marker = '*' if a == best_a and cell == 0 else ' '
        row += f"{q_vals[a]:>7.2f}{marker} "
    row += f"  {best_a_name:>10}  {cell_type}"
    print(row)

print()
print('* = Best action from that state (used by greedy policy)')

## Summary

What we built from scratch:

| Component | Implementation |
|-----------|---------------|
| `MazeEnv` | Grid world with states, actions, rewards |
| `QLearningAgent.select_action` | Epsilon-greedy policy |
| `QLearningAgent.update` | Bellman Q-update equation |
| `QLearningAgent.decay_epsilon` | Exploration → exploitation shift |
| Training loop | Episode-based rollout with Q-updates |
| Visualizations | Training curves, Q-heatmaps, policy arrows, path plot |
| Ablation | Effect of α and γ on convergence speed |

**Next steps:**
- Try a larger maze (8×8, 16×16) and observe Q-table explosion → motivation for Deep Q-Networks (DQN)
- Add stochastic transitions (wind that randomly shifts the robot) → more realistic RL
- Implement SARSA (on-policy TD) and compare with Q-learning (off-policy TD)